# Lyrics Dreamer Project

In [1]:
import os

import torch
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForCausalLM,
    # AutoModelWithLMHead,
    DataCollatorForLanguageModeling,
    Trainer, 
    TrainingArguments,
)
from datasets import load_dataset, list_datasets, load_from_disk, DatasetDict, Dataset
from pytorch_lightning.loggers import WandbLogger
import wandb

In [2]:
available_ds = [ds for ds in list_datasets() if ds.startswith('huggingartists/')]
available_artists = [ds.split('/')[-1] for ds in available_ds]
print(available_artists)

['100-gecs', '21-savage', '25-17', '50-cent', '5nizza', '5opka', '6ix9ine', 'aaron-watson', 'abba', 'adele', 'agata-christie', 'aikko', 'aimer', 'ajr', 'alan-walker', 'andre-3000', 'arash', 'architects', 'arctic-monkeys', 'ariana-grande', 'ariya', 'armin-van-buuren', 'as-i-lay-dying', 'asdfgfa', 'asper-x', 'baklan', 'big-baby-tape', 'big-russian-boss', 'bill-wurtz', 'billie-eilish', 'billy-talent', 'bladee', 'bob-dylan', 'bones', 'booker', 'boris-grebenshikov', 'braii', 'bring-me-the-horizon', 'bruce-springsteen', 'bryan-adams', 'burzum', 'bushido-zho', 'cardi-b', 'chester-bennington', 'chief-keef', 'cocomelon', 'coin', 'coldplay', 'dababy', 'david-bowie', 'ddt', 'death-grips', 'deep-purple', 'denderty', 'dermot-kennedy', 'dj-artem-artemov', 'doja-cat', 'drake', 'dua-lipa', 'duran-duran', 'dzhizus', 'ed-sheeran', 'egor-kreed', 'egor-letov', 'elton-john', 'eminem', 'enigma', 'enya', 'epic-rap-battles-of-history', 'face', 'fascinoma', 'fear-factory', 'florence-the-machine', 'freddie-dred

In [3]:
ARTISTS = [
    'radiohead', 'muse', 
    'the-beatles', 'tom-waits', 'eminem', 'nirvana', 'taylor-swift', 'bob-dylan'
]

In [4]:
os.makedirs("./data_raw", exist_ok=True)
os.makedirs("./data_clean", exist_ok=True)

In [5]:
!dir data_raw
!dir data_clean

 Volume in drive D is Fichiers & Programmes
 Volume Serial Number is 2CA8-ED5D

 Directory of D:\experiments\lyrics\data_raw

25/04/2023  19:22    <DIR>          .
26/04/2023  01:11    <DIR>          ..
25/04/2023  19:22    <DIR>          bob-dylan
25/04/2023  19:22    <DIR>          eminem
25/04/2023  19:22    <DIR>          muse
25/04/2023  19:22    <DIR>          nirvana
25/04/2023  19:22    <DIR>          oasis
25/04/2023  19:22    <DIR>          radiohead
25/04/2023  19:22    <DIR>          taylor-swift
25/04/2023  19:22    <DIR>          the-beatles
25/04/2023  19:22    <DIR>          tom-waits
               0 File(s)              0 bytes
              11 Dir(s)  623,493,947,392 bytes free
 Volume in drive D is Fichiers & Programmes
 Volume Serial Number is 2CA8-ED5D

 Directory of D:\experiments\lyrics\data_clean

26/04/2023  00:55    <DIR>          .
26/04/2023  01:11    <DIR>          ..
26/04/2023  00:55    <DIR>          arctic-monkeys
25/04/2023  19:24    <DIR>          bo

OK, so we get one row per song. Note that we also get the song listing for each record they made. Maybe we should clean that up.

In [6]:
def get_and_clean_data(artists, raw_data_root: str = './data_raw', clean_data_root: str = './data_clean'):
    for artist in artists:
        print(f"Processing {artist}")
        ds = load_dataset(f"huggingartists/{artist}")
        # Remove song listings
        ds['train'] = ds['train'].filter(lambda x: not x['text'].startswith('1'))
        # Remote empty sentences
        # ds['train'] = ds['train'].filter(lambda x: len(x['text']) > 0)
        ds.save_to_disk(os.path.join(clean_data_root, artist))

In [7]:
get_and_clean_data(ARTISTS, './data_raw')

Processing radiohead


Found cached dataset radiohead (C:/Users/lucfr/.cache/huggingface/datasets/huggingartists___radiohead/default/1.0.0/d37c36412fe443a2baba285a9d3ab4e8e4df79db8feb93b2f775c3dacfeca9eb)


  0%|          | 0/1 [00:00<?, ?it/s]

Loading cached processed dataset at C:\Users\lucfr\.cache\huggingface\datasets\huggingartists___radiohead\default\1.0.0\d37c36412fe443a2baba285a9d3ab4e8e4df79db8feb93b2f775c3dacfeca9eb\cache-bfda157b56a97188.arrow


Saving the dataset (0/1 shards):   0%|          | 0/485 [00:00<?, ? examples/s]

...

Processing bob-dylan


Found cached dataset bob-dylan (C:/Users/lucfr/.cache/huggingface/datasets/huggingartists___bob-dylan/default/1.0.0/704607c4ad7d550e18f762fbd8e17fd98d27fe7f1bbdce555c3d1555d1890bc6)


  0%|          | 0/1 [00:00<?, ?it/s]

Loading cached processed dataset at C:\Users\lucfr\.cache\huggingface\datasets\huggingartists___bob-dylan\default\1.0.0\704607c4ad7d550e18f762fbd8e17fd98d27fe7f1bbdce555c3d1555d1890bc6\cache-4b0debe4c54fb7da.arrow


Saving the dataset (0/1 shards):   0%|          | 0/2240 [00:00<?, ? examples/s]

In [18]:
checkpoint = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenizer.pad_token = tokenizer.eos_token
block_size = int(tokenizer.model_max_length / 4)
model = AutoModelForCausalLM.from_pretrained(checkpoint)

In [9]:
tokenizer.special_tokens_map

{'bos_token': '<|endoftext|>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<|endoftext|>',
 'pad_token': '<|endoftext|>'}

In [15]:
temp = dict()
tokenized_datasets = dict()
batched_datasets = dict()

def tokenize_function(examples):
    return tokenizer(
        examples['text'], 
        # truncation=True,
        # max_length=256,
        padding='max_length',
        # return_overflowing_tokens=True,
        # stride=16,
    )

def group_inputs(examples):
    
    concat_inputs = {k: [x for x in examples[k]] + [] for k in examples.keys()}
    total_length = len(concat_inputs[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concat_inputs.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

for artist in ARTISTS:
    dataset = load_from_disk(os.path.join('./data_clean', artist))
    tokenized_datasets[artist] = dataset.map(tokenize_function, batched=True, remove_columns=['text'])
    batched_datasets[artist] = tokenized_datasets[artist].map(group_inputs, batched=True, num_proc=1)

Loading cached processed dataset at D:\experiments\lyrics\data_clean\radiohead\train\cache-04d13c95c738cfd9.arrow
Loading cached processed dataset at D:\experiments\lyrics\data_clean\radiohead\train\cache-e6efa9c9bf8ea5fd.arrow
Loading cached processed dataset at D:\experiments\lyrics\data_clean\muse\train\cache-5374d463a82d0759.arrow
Loading cached processed dataset at D:\experiments\lyrics\data_clean\muse\train\cache-3823df59fa9a86de.arrow
Loading cached processed dataset at D:\experiments\lyrics\data_clean\the-beatles\train\cache-6bfcb34053aa7b6d.arrow


Map:   0%|          | 0/873 [00:00<?, ? examples/s]

Loading cached processed dataset at D:\experiments\lyrics\data_clean\tom-waits\train\cache-c4a8075a6feefde0.arrow


Map:   0%|          | 0/679 [00:00<?, ? examples/s]

Map:   0%|          | 0/1277 [00:00<?, ? examples/s]

Map:   0%|          | 0/1277 [00:00<?, ? examples/s]

Loading cached processed dataset at D:\experiments\lyrics\data_clean\nirvana\train\cache-391e16df98a7926a.arrow
Loading cached processed dataset at D:\experiments\lyrics\data_clean\nirvana\train\cache-57b00b15cc70cb48.arrow
Loading cached processed dataset at D:\experiments\lyrics\data_clean\taylor-swift\train\cache-48a2e7d43ece2c55.arrow
Loading cached processed dataset at D:\experiments\lyrics\data_clean\taylor-swift\train\cache-168e10ae0386926d.arrow
Loading cached processed dataset at D:\experiments\lyrics\data_clean\bob-dylan\train\cache-0651bab3486a45f5.arrow
Loading cached processed dataset at D:\experiments\lyrics\data_clean\bob-dylan\train\cache-035928bf9ba93931.arrow


In [16]:
for artist in ARTISTS:
    print(artist, [len(x) for x in batched_datasets[artist]['train']['input_ids']])

radiohead [256]
muse [256]
the-beatles [256, 256, 256]
tom-waits [256, 256]
eminem [256, 256, 256, 256]
nirvana [256]
taylor-swift [256, 256]
bob-dylan [256, 256, 256, 256, 256, 256]


In [17]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, return_tensors='pt')

In [13]:
# wandb.init(project="lyrics-dreamer", name='sequential-training')
# logger = WandbLogger(save_dir='./logs')

for artist, dataset in batched_datasets.items():
    print(f"Training on {artist}:")
    training_args = TrainingArguments(
        output_dir=f'./checkpoints/{artist}',
        per_device_train_batch_size=1,
        learning_rate=2e-5,
        push_to_hub=False,
        no_cuda=False,
        fp16=True,
        # logging_steps=10,
        num_train_epochs=1,
        weight_decay=1e-3,
    )
    trainer = Trainer(
        model=model,  # TODO: try with a different model for each dataset
        args=training_args,
        train_dataset=dataset['train'],
        data_collator=data_collator,
    )
    trainer.train()

Training on radiohead:


D:\envs\nlp\Lib\site-packages\transformers\optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
wandb: Currently logged in as: lucfrachon. Use `wandb login --relogin` to force relogin


You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.20 GiB (GPU 0; 15.99 GiB total capacity; 13.94 GiB already allocated; 0 bytes free; 14.18 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
torch.cuda.empty_cache()

In [ ]:
! "C:\Windows\System32\DriverStore\FileRepository\nvrfi.inf_amd64_41faa201538e6451\nvidia-smi.exe"